<div style="background: linear-gradient(135deg, #1a1a2e 0%, #2d0a16 100%); padding: 32px; border-radius: 12px; text-align: center; font-family: monospace;">
<h1 style="color: #FFD700; font-size: 2.6em; margin: 0; letter-spacing: 3px;">🌿 THE NEIGHBOR GAMES 🌿</h1>
<h2 style="color: #ffffff; font-size: 1.2em; margin: 10px 0 0 0; font-weight: normal; letter-spacing: 1px;">k-Nearest Neighbor Prediction Championship</h2>
<p style="color: #aaaacc; margin: 10px 0 0 0; font-size: 0.9em;">Elements of Data Science · Temple University</p>
<p style="color: #FFD700; margin: 8px 0 0 0; font-size: 0.85em; font-style: italic;">"May the best features be ever in your favor."</p>
</div>

---

### 🏆 Competition Overview

Tune your **k-Nearest Neighbor model** to predict alcohol boiling points as accurately as possible.  
The team with the **lowest RMSE wins** — and the **Prize Round** tests your model on molecules it has *never seen*.

| Round | Task | What you do |
|---|---|---|
| 🌿 Tribute Round | Shared baseline — run as-is | Execute only |
| ⚔️ Arena Round | Tune `k` and features | **Fill in a function call** |
| 🎯 Prize Round | Predict bp of real molecules | **Build feature arrays + compute score** |
| 🌟 Wildcard | Inverse-distance weighting | Execute only |

> 💡 **Strategy note:** Features with very different numerical scales skew distance calculations — you'll standardize the data first, which requires **filling in the z-score formula**.


### 📛 Enter your team name

In [ ]:
team_name = "..."   # e.g. "District 4: The Alkanol Alliance"

---
### ⚙️ Setup — Run this first

In [ ]:
import numpy as np
from datascience import *
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('ggplot')
Temple_color = '#9E1B34'
Gold_color   = '#FFD700'
print("✅ Modules loaded")

---
### 🔧 Your Toolkit — the same functions from lab

In [ ]:
def distance(pt1, pt2):
    """Euclidean distance between two points (arrays)."""
    return np.sqrt(sum((pt1 - pt2) ** 2))

def row_distance(row1, row2):
    """Distance between two Table rows."""
    return distance(np.array(row1), np.array(row2))

def distances(training, test, target, features):
    """Append Distance column to training for each row vs. test point."""
    dists = []
    attributes = training.select(features)
    for row in attributes.rows:
        dists.append(row_distance(row, test))
    return training.with_column('Distance', dists)

def closest(training, test, k, target, features):
    """Return the k closest training rows to a test point."""
    return distances(training, test, target, features).sort('Distance').take(np.arange(k))

def predict_knn(row, train, test, k=5, pr=False):
    """Predict target by averaging the k nearest neighbors."""
    if pr:
        print(f'Predicting for row={row}, k={k}, features={features}')
    return np.average(
        closest(train, test.select(features).row(row), k, target, features).column(target[0])
    )

def predict_knn_weighted(row, train, test, k=5, pr=False):
    """Predict target using inverse-distance weighting of k neighbors."""
    dist_table = closest(train, test.select(features).row(row), k, target, features)
    weights = 1 / (dist_table['Distance'] + 1e-9)
    return np.sum(dist_table[target[0]] * weights) / np.sum(weights)

print("✅ All tools loaded!")

---
### 📂 Load Data & Create the Competition Split

In [ ]:
ROH_data = Table().read_table('data/ROH_data.csv')
print("Columns:", ROH_data.labels)
print("Rows:   ", ROH_data.num_rows)
ROH_data.show(5)

In [ ]:
# ⚠️  Do NOT change the seed — fixed for fairness
np.random.seed(42)
shuffled  = ROH_data.sample(with_replacement=False)
split_n   = int(0.75 * shuffled.num_rows)
train_raw = shuffled.take(np.arange(split_n))
test_raw  = shuffled.take(np.arange(split_n, shuffled.num_rows))
print(f"Training rows : {train_raw.num_rows}")
print(f"Test rows     : {test_raw.num_rows}")

---
### 📏 Why We Must Standardize Features

Euclidean distance adds squared differences across all features:

$$d = \sqrt{(\Delta MW)^2 + (\Delta\text{degree})^2 + (\Delta\text{carbons})^2}$$

If MW ranges from ~30 to ~200 while carbons ranges from 1 to ~10, the **MW term dominates** — `degree` and `carbons` barely influence the result.

**Solution:** Convert each feature to **standard units** (z-score) so they all have mean ≈ 0 and std ≈ 1:

$$z = \frac{x - \mu}{\sigma}$$

> ⚠️ **Key rule:** Always compute $\mu$ and $\sigma$ from the **training data only**, then apply those same values to the test set. Never let the model peek at test statistics.


In [ ]:
all_features = ["MW", "degree", "carbons"]

print("Feature ranges in training data:")
print(f"{'Feature':<12} {'Min':>8} {'Max':>8} {'Mean':>8} {'Std':>8}")
print("─" * 50)
for f in all_features:
    col = train_raw.column(f)
    print(f"{f:<12} {np.min(col):>8.2f} {np.max(col):>8.2f} {np.mean(col):>8.2f} {np.std(col):>8.2f}")
print()
print("👆 Notice how MW dwarfs the other features in scale.")

---
<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Coding Challenge 1 of 4 — The Standardization Formula</strong><br>
Complete the z-score formula inside the loop below.<br>
Recall: &nbsp; $z = \dfrac{x - \mu}{\sigma}$ &nbsp; where $\mu$ is the mean and $\sigma$ is the standard deviation.
</div>


In [ ]:
# Compute and store training-set statistics for each feature.
# These will travel with the model and be reused in the Prize Round.
train_stats = {}   # { feature_name: (mean, std) }

for f in all_features:
    col   = train_raw.column(f)
    mu    = np.mean(col)
    sigma = np.std(col)
    train_stats[f] = (mu, sigma)

# Build standardized train and test tables.
# bp (the target) is NOT standardized — we predict it in original Kelvin units.
train = train_raw.select('bp')
test  = test_raw.select('bp')
# hint: x is the Table column   (train_raw.column(f) - mu) / sigma
for f in all_features:
    mu, sigma = train_stats[f]
    # 🖊️  Fill in the z-score formula for the training column:
    train = train.with_columns(f, ...) # 
    # 🖊️  Apply the SAME mu and sigma to the test column:
    test  = test.with_columns( f, ...)

print("Standardized training data (first 5 rows):")
train.show(5)

In [ ]:
# Verify: each feature should now have mean ≈ 0 and std ≈ 1
print(f"{'Feature':<12} {'Mean':>10} {'Std':>10}   (target: ≈0, ≈1)")
print("─" * 48)
for f in all_features:
    col = train.column(f)
    print(f"{f:<12} {np.mean(col):>10.4f} {np.std(col):>10.4f}")

---
#### 📊 Scoring function — run as-is

In [ ]:
def compute_score(predictions, label='Score'):
    """Compute RMSE vs. test targets. Lower = better."""
    actual = test.column('bp')
    preds  = np.array(predictions)
    if len(preds) != len(actual):
        print(f"⚠️  Need {len(actual)} predictions, got {len(preds)}")
        return None
    rmse = np.sqrt(np.mean((actual - preds) ** 2))
    fill = int(max(0, 28 - rmse / 2))
    bar  = '█' * fill + '░' * (28 - fill)
    print(f"{'─'*52}")
    print(f"  {label}")
    print(f"  RMSE = {rmse:.3f} K   [{bar}]")
    print(f"{'─'*52}")
    return round(rmse, 3)

---
## 🌿 Tribute Round — Shared Baseline
Run this **exactly as written**. This is everyone's starting point.

In [ ]:
features = ["MW"]
target   = ["bp"]
k        = 5

tribute_predictions = [predict_knn(i, train, test, k=k) for i in np.arange(test.num_rows)]
tribute_rmse = compute_score(tribute_predictions, label="Tribute Baseline (MW only, k=5)")

In [ ]:
actual_bp = test_raw.column('bp')
plt.figure(figsize=(6, 5))
plt.scatter(actual_bp, tribute_predictions, color=Temple_color, alpha=0.7, edgecolors='white', s=60)
diag = np.linspace(min(actual_bp), max(actual_bp), 100)
plt.plot(diag, diag, '--', color=Gold_color, linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual bp (K)', fontsize=12)
plt.ylabel('Predicted bp (K)', fontsize=12)
plt.title(f'Tribute Round  |  RMSE = {tribute_rmse:.2f} K', fontsize=11)
plt.legend(); plt.tight_layout(); plt.show()

---
## ⚔️ Arena Round — Tune Your Model!

Choose your **`features`** and **`k`**, then complete the prediction call.

**Available features (all pre-standardized):** `"MW"`, `"degree"`, `"carbons"`

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Coding Challenge 2 of 4 — Make Predictions</strong><br>
Complete the list comprehension so it calls <code>predict_knn</code> for every row in the test set.<br>
<code>predict_knn</code> signature: &nbsp;<code>predict_knn(row, train, test, k)</code>
</div>


In [ ]:
# ── ⚔️  STEP 1: choose your settings ─────────────────────────────
features = ["MW", "degree"]    # try adding "carbons"
k        = 5                   # try 3, 7, 10, 15 …
target   = ["bp"]

# ── ⚔️  STEP 2: fill in the predict_knn call ──────────────────────
arena_predictions = [predict_knn(..., ..., ..., k=k) for i in np.arange(test.num_rows)]

arena_rmse = compute_score(arena_predictions, label=f"Arena: features={features}, k={k}")

In [ ]:
actual_bp = test_raw.column('bp')
plt.figure(figsize=(6, 5))
plt.scatter(actual_bp, arena_predictions, color='steelblue', alpha=0.7, edgecolors='white', s=60)
diag = np.linspace(min(actual_bp), max(actual_bp), 100)
plt.plot(diag, diag, '--', color=Gold_color, linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual bp (K)', fontsize=12)
plt.ylabel('Predicted bp (K)', fontsize=12)
plt.title(f'Arena Round  |  k={k}, features={features}\nRMSE = {arena_rmse:.2f} K', fontsize=11)
plt.legend(); plt.tight_layout(); plt.show()

#### 🔬 k-sweep — run as-is to find your best k

In [ ]:
features_sweep = ["MW", "degree"]    # ← paste your best features
k_values  = np.arange(1, 21)
rmse_list = []

for kk in k_values:
    preds = [predict_knn(i, train, test, k=kk) for i in np.arange(test.num_rows)]
    rmse_list.append(np.sqrt(np.mean((test.column('bp') - np.array(preds)) ** 2)))

best_k = k_values[np.argmin(rmse_list)]
plt.figure(figsize=(7, 4))
plt.plot(k_values, rmse_list, marker='o', color=Temple_color, linewidth=2, markersize=6)
plt.axvline(best_k, color=Gold_color, linestyle='--', label=f'Best k = {best_k}')
plt.xlabel('k (number of neighbors)', fontsize=12)
plt.ylabel('Test RMSE (K)', fontsize=12)
plt.title(f'k vs. RMSE  |  features = {features_sweep}', fontsize=11)
plt.legend(); plt.tight_layout(); plt.show()
print(f"Best k = {best_k}  →  RMSE = {min(rmse_list):.3f} K")

---
## 🏆 Prize Round — Predict Real Molecules!

Predict the boiling point of two common alcohols **not in the dataset**.  
The team whose predictions are closest to the real values wins the prize!

| Molecule | Common use | MW (g/mol) | Degree | Carbons | Actual bp |
|---|---|---|---|---|---|
| **Ethanol** | Hand sanitizer, beverages | 46.07 | 1 | 2 | 351.4 K |
| **Isopropanol (IPA)** | Rubbing alcohol, solvent | 60.10 | 2 | 3 | 355.4 K |

> ⚠️ **Critical:** Apply the **training-set** $\mu$ and $\sigma$ from `train_stats` to standardize these new molecules — the same values used to standardize the training data.


#### Step 1 — Set your best settings from the Arena Round

In [ ]:
features = ["MW", "degree"]   # ← your best features
k        = 5                  # ← your best k
target   = ["bp"]

#### Step 2 — Standardize the new molecules

Ethanol is done for you as a worked example.

The z-score for a single raw value $x$ using training statistics is:

$$z = \frac{x - \mu_{\text{train}}}{\sigma_{\text{train}}}$$


<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Coding Challenge 3 of 4 — Standardize Isopropanol</strong><br>
Ethanol's standardized array is built for you below. Using the same pattern, build <code>ipa_std</code> for isopropanol.<br>
Raw values: MW = 60.10, degree = 2, carbons = 3. Remember to use <code>train_stats</code> and to match the order of <code>features</code>.
</div>


In [ ]:
# ── Ethanol — worked example ──────────────────────────────────────
# Raw feature values
ethanol_raw = {"MW": 46.07, "degree": 1, "carbons": 2}

# Standardize each feature using training statistics
ethanol_std = np.array([
    (ethanol_raw[f] - train_stats[f][0]) / train_stats[f][1]
    for f in features
])
print("Ethanol standardized:", dict(zip(features, ethanol_std.round(4))))

# ── Isopropanol — your turn ───────────────────────────────────────
ipa_raw = {"MW": 60.10, "degree": 2, "carbons": 3}

# 🖊️  Build ipa_std using the same pattern as ethanol_std above:
ipa_std = ...

print("IPA standardized:    ", dict(zip(features, ipa_std.round(4))))

#### Step 3 — Predict and display neighbors

In [ ]:
# Ethanol prediction
ethanol_neighbors = closest(train, ethanol_std, k, target, features)
print(f"{'─'*40}")
print(f"  k={k} nearest neighbors to ethanol")
print(f"{'─'*40}")
ethanol_neighbors.show()
ethanol_bp_pred = np.average(ethanol_neighbors.column(target[0]))
ETHANOL_ACTUAL  = 351.4
print(f"Predicted : {ethanol_bp_pred:.1f} K  |  Actual : {ETHANOL_ACTUAL} K  |  Error : {abs(ethanol_bp_pred-ETHANOL_ACTUAL):.1f} K")

In [ ]:
# IPA prediction
ipa_neighbors = closest(train, ipa_std, k, target, features)
print(f"{'─'*40}")
print(f"  k={k} nearest neighbors to isopropanol")
print(f"{'─'*40}")
ipa_neighbors.show()
ipa_bp_pred = np.average(ipa_neighbors.column(target[0]))
IPA_ACTUAL  = 355.4
print(f"Predicted : {ipa_bp_pred:.1f} K  |  Actual : {IPA_ACTUAL} K  |  Error : {abs(ipa_bp_pred-IPA_ACTUAL):.1f} K")

<div style="background:#fff8e1; border-left:4px solid #FFD700; padding:12px 16px; border-radius:0 8px 8px 0;">
<strong>🖊️ Coding Challenge 4 of 4 — Compute the Prize Score</strong><br>
The prize score is the <strong>average absolute prediction error</strong> across both molecules.<br>
Fill in the formula: &nbsp;<code>prize_score = (ethanol_error + ipa_error) / ...</code>
</div>


In [ ]:
ethanol_error = abs(ethanol_bp_pred - ETHANOL_ACTUAL)
ipa_error     = abs(ipa_bp_pred     - IPA_ACTUAL)

# 🖊️  Compute the average absolute error (prize score):
prize_score = (ethanol_error + ipa_error) / ...

print("=" * 52)
print(f"  🏆  PRIZE ROUND — {team_name}")
print("=" * 52)
print(f"  Ethanol error     : {ethanol_error:.2f} K")
print(f"  Isopropanol error : {ipa_error:.2f} K")
print(f"  ─────────────────────────────────────────")
print(f"  Prize Score (avg) : {prize_score:.2f} K  ← lower wins!")
print("=" * 52)

# Visualisation
fig, ax = plt.subplots(figsize=(6, 4))
x, w = np.arange(2), 0.3
ax.bar(x-w/2, [ETHANOL_ACTUAL, IPA_ACTUAL], w, label='Actual',    color=Gold_color,   edgecolor='black', linewidth=0.5)
ax.bar(x+w/2, [ethanol_bp_pred, ipa_bp_pred], w, label='Predicted', color=Temple_color, edgecolor='black', linewidth=0.5, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(['Ethanol\n(actual 351.4 K)', 'Isopropanol\n(actual 355.4 K)'], fontsize=11)
ax.set_ylabel('Boiling Point (K)', fontsize=12)
ax.set_title(f'Prize Round: {team_name}', fontsize=11)
ax.legend(); plt.tight_layout(); plt.show()

---
## 🌟 Wildcard Round — Inverse-Distance Weighting
Closer neighbors get more influence. Does weighting help on standardized data?

In [ ]:
features = ["MW", "degree"]   # ← your best features
k        = 5                  # ← your best k
target   = ["bp"]

wildcard_predictions = [predict_knn_weighted(i, train, test, k=k) for i in np.arange(test.num_rows)]
wildcard_rmse = compute_score(wildcard_predictions, label=f"Wildcard (weighted): features={features}, k={k}")

---
## 💬 Reflection
Double-click to answer:

1. **In Challenge 1, why do we apply the training mean and std to the test set rather than computing new statistics from the test set?**
2. **In Challenge 2, what does the `i` variable represent in the list comprehension?**
3. **In Challenge 3, why can't we just use `standard_units(np.array([46.07]))` to standardize ethanol's MW?**
4. **Did adding more features (degree, carbons) improve your Arena RMSE after standardization? Was the improvement bigger or smaller than you expected?**


*Your answers here:*

1. 

2. 

3. 

4. 

---
## 📋 Final Submission — Screenshot and share!

In [ ]:
print("=" * 54)
print(f"  🌿  THE NEIGHBOR GAMES — {team_name}")
print("=" * 54)
try: print(f"  Tribute Baseline  : {tribute_rmse:.3f} K")
except: pass
try: print(f"  Arena Round RMSE  : {arena_rmse:.3f} K   features={features}, k={k}")
except: pass
try: print(f"  Prize Score (avg) : {prize_score:.2f} K  ← lower wins prize")
except: pass
try: print(f"  Wildcard RMSE     : {wildcard_rmse:.3f} K")
except: pass
print("=" * 54)
print("  📸  Screenshot this and report to instructor!")